# Dataset óptico (Sentinel-2 / Landsat) para las mismas zonas y fechas que las manchas de petróleo (Cerulean SkyTruth, Sentinel-1)

Este notebook toma el `geojson` de manchas de petróleo (SAR, Sentinel-1, dataset Cerulean SkyTruth) y arma un **catálogo de imágenes ópticas** (Sentinel-2 SR y Landsat 8/9 SR Collection 2) que:

- cubren la **misma zona** (bounding box de cada mancha / escena S1, con un buffer),
- están cerca en el **tiempo** (± N días respecto a `slick_timestamp`), y
- tienen **baja cobertura de nubes**.

El catálogo resultante (CSV + GeoJSON) tiene, por cada escena Sentinel-1 con mancha detectada, la mejor imagen S2 y la mejor imagen Landsat disponibles, con su `id` de Earth Engine, fecha, % de nubes y un link de thumbnail. También incluye código para exportar los recortes (chips) como GeoTIFF a Google Drive.

Corre esto en **Google Colab**. Necesitás una cuenta de [Google Earth Engine](https://code.earthengine.google.com/register) (gratuita) y un *Cloud Project* asociado.

## 1. Instalación y autenticación

In [ ]:
!pip install -q earthengine-api geemap geopandas


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 28.6 MB/s eta 0:00:00


In [ ]:
import ee
import geemap

# Reemplazá 'tu-proyecto-gee' por el ID de tu Google Cloud Project habilitado para Earth Engine
# (Project Settings -> Project ID en https://code.earthengine.google.com)
EE_PROJECT = 'TU_PROYECTO_EE'

ee.Authenticate()
ee.Initialize(project=EE_PROJECT)


## 2. Subir y cargar el dataset de manchas (Sentinel-1)

Subí a Colab el archivo `golfo_persico_fixed.geojson` (el que ya corregimos, con geometría válida).

In [ ]:
from google.colab import files
uploaded = files.upload()  # elegí golfo_persico_fixed.geojson
slick_path = list(uploaded.keys())[0]


Saving cerulean_combinado.csv to cerulean_combinado.csv


In [ ]:
import geopandas as gpd
import pandas as pd
import json
from shapely.geometry import shape, GeometryCollection
import ast # Import the ast module

slicks = gpd.read_file(slick_path)

# Check if 'centerlines' column exists and create 'geometry' from it
if 'centerlines' in slicks.columns:
    def parse_geometry(x):
        if pd.isna(x):
            return None
        parsed_obj = ast.literal_eval(x)
        if isinstance(parsed_obj, dict):
            obj_type = parsed_obj.get('type')
            if obj_type == 'FeatureCollection':
                # If it's a FeatureCollection, extract geometries from all features
                geoms = [shape(f['geometry']) for f in parsed_obj.get('features', []) if 'geometry' in f]
                return GeometryCollection(geoms) if geoms else None
            elif obj_type == 'Feature':
                # If it's a Feature, extract its geometry
                return shape(parsed_obj.get('geometry'))
            else:
                # Assume it's a direct geometry object (e.g., Point, LineString, Polygon)
                return shape(parsed_obj)
        return None # Fallback for unexpected types

    slicks['geometry'] = slicks['centerlines'].apply(parse_geometry)
    # Convert the DataFrame to a GeoDataFrame
    slicks = gpd.GeoDataFrame(slicks, geometry='geometry', crs='EPSG:4326')
    print("Successfully created 'geometry' column from 'centerlines' (parsed as Python literal).")
elif 'geometry' not in slicks.columns:
    # This block handles cases where 'centerlines' is also missing
    print("Warning: 'geometry' column not found and 'centerlines' column is missing. Cannot create GeoDataFrame geometry.")

slicks['slick_timestamp'] = pd.to_datetime(slicks['slick_timestamp'])
print(f"{len(slicks)} manchas, {slicks['s1_scene_id'].nunique()} escenas Sentinel-1 únicas")
print("Rango de fechas:", slicks['slick_timestamp'].min(), "-", slicks['slick_timestamp'].max())
slicks[['id', 's1_scene_id', 'slick_timestamp', 'machine_confidence']].head()

Successfully created 'geometry' column from 'centerlines' (parsed as Python literal).
10000 manchas, 1985 escenas Sentinel-1 únicas
Rango de fechas: 2023-01-02 03:14:46+00:00 - 2025-05-30 03:37:45+00:00


,id,s1_scene_id,slick_timestamp,machine_confidence
0,3000373,S1A_IW_GRDH_1SDV_20240908T143412_20240908T1434...,2024-09-08 14:34:12+00:00,0.8634161949157715
1,3000374,S1A_IW_GRDH_1SDV_20240908T143412_20240908T1434...,2024-09-08 14:34:12+00:00,0.7584561705589294
2,3000375,S1A_IW_GRDH_1SDV_20240908T143412_20240908T1434...,2024-09-08 14:34:12+00:00,0.7588280439376831
3,3000376,S1A_IW_GRDH_1SDV_20240908T143412_20240908T1434...,2024-09-08 14:34:12+00:00,0.6815117597579956
4,3000377,S1A_IW_GRDH_1SDV_20240908T143412_20240908T1434...,2024-09-08 14:34:12+00:00,0.726639986038208


## 3. Agrupar por escena Sentinel-1

Como varias manchas provienen de la misma escena S1 (misma fecha/pasada), buscamos una imagen óptica **por escena**, no por mancha individual, para no repetir búsquedas innecesarias en Earth Engine.

In [ ]:
# Un registro por escena S1: fecha + bbox combinado de todas sus manchas (+ buffer)
BUFFER_KM = 10  # margen alrededor del bbox de las manchas, en km

def bbox_with_buffer(grp_gdf, buffer_km):
    # grp_gdf es ahora un GeoDataFrame
    minx, miny, maxx, maxy = grp_gdf.total_bounds

    # aproximación: 1 grado ~ 111 km
    buf_deg = buffer_km / 111.0
    return (minx - buf_deg, miny - buf_deg,
            maxx + buf_deg, maxy + buf_deg)

scenes = []
for scene_id, grp_df_pandas in slicks.groupby('s1_scene_id'):
    # Convertir el DataFrame de pandas agrupado a un GeoDataFrame para un manejo adecuado de la geometría
    # Asegurarse de que la columna 'geometry' existe antes de crear el GeoDataFrame
    if 'geometry' not in grp_df_pandas.columns:
        print(f"Advertencia: la columna 'geometry' no se encontró para scene_id {scene_id}. Omitiendo este grupo.")
        continue

    # Pasar la CRS del GeoDataFrame 'slicks' original
    grp_gdf = gpd.GeoDataFrame(grp_df_pandas, geometry='geometry', crs=slicks.crs)

    minx, miny, maxx, maxy = bbox_with_buffer(grp_gdf, BUFFER_KM)
    scenes.append({
        's1_scene_id': scene_id,
        'slick_timestamp': grp_gdf['slick_timestamp'].iloc[0],
        'n_slicks': len(grp_gdf),
        'minx': minx, 'miny': miny, 'maxx': maxx, 'maxy': maxy,
    })

scenes_df = pd.DataFrame(scenes).sort_values('slick_timestamp').reset_index(drop=True)
print(f"{len(scenes_df)} escenas S1 a emparejar con óptico")
scenes_df.head()

1985 escenas S1 a emparejar con óptico


,s1_scene_id,slick_timestamp,n_slicks,minx,miny,maxx,maxy
0,S1A_IW_GRDH_1SDV_20230102T031446_20230102T0315...,2023-01-02 03:14:46+00:00,9,38.614078,19.266509,40.295876,20.516272
1,S1A_IW_GRDH_1SDV_20230102T031605_20230102T0316...,2023-01-02 03:16:05+00:00,2,39.592565,15.703471,39.855032,15.921302
2,S1A_IW_GRDH_1SDV_20230102T154717_20230102T1547...,2023-01-02 15:47:17+00:00,1,35.649985,24.408803,35.959588,24.618953
3,S1A_IW_GRDH_1SDV_20230102T154746_20230102T1548...,2023-01-02 15:47:46+00:00,4,34.010352,26.492813,35.424053,27.353830
4,S1A_IW_GRDH_1SDV_20230105T033724_20230105T0337...,2023-01-05 03:37:24+00:00,2,34.139915,26.949417,35.713139,27.368504


## 4. Funciones de búsqueda de imagen óptica

Para cada escena S1 buscamos, dentro de una ventana de ± `DAYS_WINDOW` días:

- la mejor imagen **Sentinel-2 SR Harmonized** (menor % de nubes), y
- la mejor imagen **Landsat 8/9 Collection 2 SR** como alternativa (útil para fechas de 2023 con menos cobertura S2, o si S2 sale muy nublado).


In [ ]:
DAYS_WINDOW = 5        # buscar óptico hasta N días antes/después de la mancha
MAX_CLOUD_S2 = 40      # % máximo de nubes aceptado en Sentinel-2 (CLOUDY_PIXEL_PERCENTAGE)
MAX_CLOUD_L8 = 40       # % máximo de nubes aceptado en Landsat (CLOUD_COVER)

S2_COLLECTION = 'COPERNICUS/S2_SR_HARMONIZED'
L8_COLLECTION = 'LANDSAT/LC08/C02/T1_L2'
L9_COLLECTION = 'LANDSAT/LC09/C02/T1_L2'


def make_aoi(row):
    return ee.Geometry.Rectangle([row['minx'], row['miny'], row['maxx'], row['maxy']])


def best_image(collection_id, aoi, center_date, days_window, cloud_prop, max_cloud):
    start = center_date - pd.Timedelta(days=days_window)
    end = center_date + pd.Timedelta(days=days_window)
    coll = (ee.ImageCollection(collection_id)
            .filterBounds(aoi)
            .filterDate(start.strftime('%Y-%m-%d'), (end + pd.Timedelta(days=1)).strftime('%Y-%m-%d'))
            .filter(ee.Filter.lt(cloud_prop, max_cloud))
            .sort(cloud_prop))
    n = coll.size().getInfo()
    if n == 0:
        return None
    img = ee.Image(coll.first())
    info = img.getInfo()
    props = info['properties']
    img_id = info['id']
    cloud = props.get(cloud_prop)
    date_str = ee.Date(props['system:time_start']).format('YYYY-MM-dd HH:mm').getInfo()

    # Asegurarse de que el Timestamp de la imagen también sea tz-aware (UTC)
    image_timestamp = pd.to_datetime(date_str).tz_localize('UTC')
    days_diff = abs((image_timestamp - center_date).total_seconds() / 86400)
    return {
        'image_id': img_id,
        'date': date_str,
        'cloud_pct': cloud,
        'days_from_slick': round(days_diff, 2),
        'n_candidates': n,
    }


def best_optical_match(row):
    aoi = make_aoi(row)
    center_date = row['slick_timestamp']

    s2 = best_image(S2_COLLECTION, aoi, center_date, DAYS_WINDOW, 'CLOUDY_PIXEL_PERCENTAGE', MAX_CLOUD_S2)

    l8 = best_image(L8_COLLECTION, aoi, center_date, DAYS_WINDOW, 'CLOUD_COVER', MAX_CLOUD_L8)
    l9 = best_image(L9_COLLECTION, aoi, center_date, DAYS_WINDOW, 'CLOUD_COVER', MAX_CLOUD_L8)
    landsat = None
    for cand in (l8, l9):
        if cand is not None and (landsat is None or cand['cloud_pct'] < landsat['cloud_pct']):
            landsat = cand

    return s2, landsat

## 5. Correr el emparejamiento sobre todas las escenas

Esto hace varias llamadas a la API de Earth Engine por escena (puede tardar). Para probar primero con pocas escenas, ajustá `N_TEST`.

In [ ]:
N_TEST = 20  # poné un número (ej. 20) para probar rápido, o None para correr todo

subset = scenes_df if N_TEST is None else scenes_df.head(N_TEST)

results = []
for i, row in subset.iterrows():
    try:
        s2, landsat = best_optical_match(row)
    except Exception as e:
        print(f"Error en escena {row['s1_scene_id']}: {e}")
        s2, landsat = None, None

    results.append({
        's1_scene_id': row['s1_scene_id'],
        's1_date': row['slick_timestamp'],
        'n_slicks': row['n_slicks'],
        'minx': row['minx'], 'miny': row['miny'], 'maxx': row['maxx'], 'maxy': row['maxy'],
        's2_image_id': s2['image_id'] if s2 else None,
        's2_date': s2['date'] if s2 else None,
        's2_cloud_pct': s2['cloud_pct'] if s2 else None,
        's2_days_from_slick': s2['days_from_slick'] if s2 else None,
        'landsat_image_id': landsat['image_id'] if landsat else None,
        'landsat_date': landsat['date'] if landsat else None,
        'landsat_cloud_pct': landsat['cloud_pct'] if landsat else None,
        'landsat_days_from_slick': landsat['days_from_slick'] if landsat else None,
    })

    if i % 25 == 0:
        print(f"{i+1}/{len(subset)} escenas procesadas")

catalog = pd.DataFrame(results)
print("Con match Sentinel-2:", catalog['s2_image_id'].notna().sum(), "/", len(catalog))
print("Con match Landsat:   ", catalog['landsat_image_id'].notna().sum(), "/", len(catalog))
catalog.head()


1/20 escenas procesadas
Con match Sentinel-2: 20 / 20
Con match Landsat:    20 / 20


,s1_scene_id,s1_date,n_slicks,minx,miny,maxx,maxy,s2_image_id,s2_date,s2_cloud_pct,s2_days_from_slick,landsat_image_id,landsat_date,landsat_cloud_pct,landsat_days_from_slick
0,S1A_IW_GRDH_1SDV_20230102T031446_20230102T0315...,2023-01-02 03:14:46+00:00,9,38.614078,19.266509,40.295876,20.516272,COPERNICUS/S2_SR_HARMONIZED/20230106T075219_20...,2023-01-06 08:04,7.461651,4.20,LANDSAT/LC08/C02/T1_L2/LC08_169047_20230102,2023-01-02 07:44,8.54,0.19
1,S1A_IW_GRDH_1SDV_20230102T031605_20230102T0316...,2023-01-02 03:16:05+00:00,2,39.592565,15.703471,39.855032,15.921302,COPERNICUS/S2_SR_HARMONIZED/20230103T074219_20...,2023-01-03 07:55,0.250274,1.19,LANDSAT/LC08/C02/T1_L2/LC08_169049_20230102,2023-01-02 07:45,2.96,0.19
2,S1A_IW_GRDH_1SDV_20230102T154717_20230102T1547...,2023-01-02 15:47:17+00:00,1,35.649985,24.408803,35.959588,24.618953,COPERNICUS/S2_SR_HARMONIZED/20221228T081341_20...,2022-12-28 08:23,0.006928,5.31,LANDSAT/LC09/C02/T1_L2/LC09_172043_20221230,2022-12-30 08:01,4.09,3.32
3,S1A_IW_GRDH_1SDV_20230102T154746_20230102T1548...,2023-01-02 15:47:46+00:00,4,34.010352,26.492813,35.424053,27.353830,COPERNICUS/S2_SR_HARMONIZED/20221228T081341_20...,2022-12-28 08:22,0.008719,5.31,LANDSAT/LC08/C02/T1_L2/LC08_173041_20221229,2022-12-29 08:06,0.00,4.32
4,S1A_IW_GRDH_1SDV_20230105T033724_20230105T0337...,2023-01-05 03:37:24+00:00,2,34.139915,26.949417,35.713139,27.368504,COPERNICUS/S2_SR_HARMONIZED/20230105T082239_20...,2023-01-05 08:32,0.011305,0.20,LANDSAT/LC09/C02/T1_L2/LC09_173041_20230106,2023-01-06 08:06,0.27,1.19


## 6. Guardar el catálogo (CSV + GeoJSON con el bbox de cada escena)

In [ ]:
from shapely.geometry import box

catalog_gdf = gpd.GeoDataFrame(
    catalog,
    geometry=[box(r.minx, r.miny, r.maxx, r.maxy) for r in catalog.itertuples()],
    crs='EPSG:4326'
)

catalog.to_csv('optical_catalog.csv', index=False)
catalog_gdf.to_file('optical_catalog.geojson', driver='GeoJSON')

from google.colab import files
files.download('optical_catalog.csv')
files.download('optical_catalog.geojson')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 7. (Opcional) Exportar los recortes ópticos como GeoTIFF a Google Drive

Ejemplo para exportar las bandas RGB + NIR de Sentinel-2 recortadas al AOI de cada escena. Ajustá `bands` y `scale` según necesites (10 m para B2/B3/B4/B8).

In [ ]:
def export_s2_chip(image_id, aoi, description, folder='oil_slick_optical', scale=10):
    img = ee.Image(image_id).select(['B4', 'B3', 'B2', 'B8'])  # RGB + NIR
    task = ee.batch.Export.image.toDrive(
        image=img.clip(aoi),
        description=description,
        folder=folder,
        fileNamePrefix=description,
        region=aoi,
        scale=scale,
        maxPixels=1e10,
    )
    task.start()
    return task

# Ejemplo: exportar los primeros 5 matches S2 del catálogo
tasks = []
for r in catalog.dropna(subset=['s2_image_id']).head(5).itertuples():
    aoi = ee.Geometry.Rectangle([r.minx, r.miny, r.maxx, r.maxy])
    desc = f"S2_{r.s1_scene_id}".replace(':', '_')[:100]
    t = export_s2_chip(r.s2_image_id, aoi, desc)
    tasks.append(t)
    print('Export iniciado:', desc)

print("Revisá el progreso en https://code.earthengine.google.com/tasks")


Export iniciado: S2_S1A_IW_GRDH_1SDV_20230102T031446_20230102T031515_046599_0595A8_D561
Export iniciado: S2_S1A_IW_GRDH_1SDV_20230102T031605_20230102T031630_046599_0595A8_CD27
Export iniciado: S2_S1A_IW_GRDH_1SDV_20230102T154717_20230102T154746_046607_0595F4_9CDA
Export iniciado: S2_S1A_IW_GRDH_1SDV_20230102T154746_20230102T154811_046607_0595F4_805B
Export iniciado: S2_S1A_IW_GRDH_1SDV_20230105T033724_20230105T033749_046643_05972C_1A0C
Revisá el progreso en https://code.earthengine.google.com/tasks


## 8. Visualización rápida con geemap (mancha S1 + imagen óptica)

In [ ]:
Map = geemap.Map()

# elegí una fila con match S2 para visualizar
row = catalog.dropna(subset=['s2_image_id']).iloc[0]
aoi = ee.Geometry.Rectangle([row.minx, row.miny, row.maxx, row.maxy])
s2_img = ee.Image(row.s2_image_id)

vis_params = {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 3000}
Map.centerObject(aoi, 11)
Map.addLayer(s2_img.clip(aoi), vis_params, f"S2 {row.s2_date}")

# manchas de esa escena, sobre el mapa
slicks_scene = slicks[slicks['s1_scene_id'] == row.s1_scene_id]
Map.add_gdf(slicks_scene, layer_name='Manchas S1', style={'color': 'red', 'fillOpacity': 0.3})

Map


## 9. Catálogo de esfuerzo: todas las pasadas de Sentinel-1 sobre el AOI (positivas y negativas)

Necesario para responder con rigor las preguntas 3 y 4 de la hoja de ruta: ¿cuánto del patrón
observado es cobertura del satélite y cuánto incidencia real?, y el modelo predictivo de
probabilidad de detección. Hasta ahora solo se cataloga óptico para las escenas S1 que **ya**
tienen una mancha detectada por Cerulean — eso no sirve para medir esfuerzo de observación,
porque por construcción el 100% de esas filas son "positivas".

Esta sección consulta la colección completa `COPERNICUS/S1_GRD` (modo IW) sobre cada zona,
tenga o no detección, la cruza con la misma rejilla espacial (`cell_id`) construida en el
notebook de limpieza (mismo `CELL_KM` y mismo bbox por zona — **deben coincidir exactamente**
para que los `cell_id` sean comparables entre notebooks) y produce el panel celda × escena y su
agregado celda-mes, con `n_pases` listo como offset (`log(n_pases)`) para el modelo GLM de la
fase 6.

Nota de escala: consultar años de escenas S1 sobre un AOI grande puede acercarse a los límites
de tamaño de respuesta de `getInfo()`. Por eso se pide en bloques de `chunk_days` — si un bloque
falla, baja `chunk_days`; para volúmenes muy grandes, la alternativa robusta es
`ee.batch.Export.table.toDrive` en vez de `getInfo()`.

In [ ]:
import numpy as np
from shapely.geometry import box, Polygon

# Deben coincidir con STUDY_AREAS y CELL_KM del notebook de limpieza
# (Copia_de_TFM_01_descarga_limpieza_exploracion_Cerulean.ipynb, sección 8.1)
# Área fijada en Golfo Pérsico.
STUDY_AREAS = {
    "golfo_persico": (47.5, 23.5, 56.5, 30.5),
}
CELL_KM = 10

FECHA_INICIO = "2023-01-01"
FECHA_FIN = pd.Timestamp.now(tz="UTC").strftime("%Y-%m-%d")

S1_COLLECTION_ID = 'COPERNICUS/S1_GRD'


def build_grid(bbox, cell_km=CELL_KM):
    minx, miny, maxx, maxy = bbox
    lat_mid = (miny + maxy) / 2
    dlat = cell_km / 111.0
    dlon = cell_km / (111.0 * np.cos(np.radians(lat_mid)))
    xs = np.arange(minx, maxx, dlon)
    ys = np.arange(miny, maxy, dlat)
    rows = []
    for i, x in enumerate(xs):
        for j, y in enumerate(ys):
            rows.append({"cell_id": f"{i}_{j}", "geometry": box(x, y, x + dlon, y + dlat)})
    return gpd.GeoDataFrame(rows, crs="EPSG:4326")


def fetch_s1_scene_catalog(aoi_bbox, start_date, end_date, chunk_days=30, verbose=True):
    """Catálogo de TODAS las escenas Sentinel-1 IW sobre aoi_bbox en el periodo,
    tengan o no detección de Cerulean. Se consulta en bloques de `chunk_days`
    para no exceder el tamaño de respuesta de Earth Engine en colecciones largas."""
    aoi = ee.Geometry.Rectangle(list(aoi_bbox))
    all_scenes = []
    cur = pd.Timestamp(start_date, tz='UTC')
    end = pd.Timestamp(end_date, tz='UTC')

    while cur < end:
        chunk_end = min(cur + pd.Timedelta(days=chunk_days), end)
        coll = (ee.ImageCollection(S1_COLLECTION_ID)
                .filterBounds(aoi)
                .filterDate(cur.strftime('%Y-%m-%d'), chunk_end.strftime('%Y-%m-%d'))
                .filter(ee.Filter.eq('instrumentMode', 'IW')))
        try:
            info = coll.getInfo()
        except Exception as e:
            print(f"  [ERROR] Bloque {cur.date()}–{chunk_end.date()} falló ({e}); prueba con chunk_days menor.")
            cur = chunk_end
            continue

        feats = info.get('features', [])
        for feat in feats:
            props = feat.get('properties', {})
            # OJO: S1_GRD no trae 'geometry' a nivel de feature (sale None).
            # La huella real de la escena está en properties['system:footprint']
            # como LinearRing, que hay que envolver como Polygon.
            footprint = props.get('system:footprint')
            if not footprint or 'coordinates' not in footprint:
                continue
            all_scenes.append({
                's1_scene_id': feat['id'].split('/')[-1],
                'timestamp_ms': props.get('system:time_start'),
                'orbit_pass': props.get('orbitProperties_pass'),
                'geometry': Polygon(footprint['coordinates']),
            })
        if verbose:
            print(f"  {cur.date()} -> {chunk_end.date()}: {len(feats)} escenas (acumulado {len(all_scenes)})")
        cur = chunk_end

    return all_scenes


exposure_panels = {}

for area_name, bbox in STUDY_AREAS.items():
    print(f"\n=== Catálogo de esfuerzo S1: {area_name} ===")
    scenes = fetch_s1_scene_catalog(bbox, FECHA_INICIO, FECHA_FIN)
    if not scenes:
        print(f"  [AVISO] Sin escenas para {area_name}.")
        continue

    scenes_gdf = gpd.GeoDataFrame(
        [{"s1_scene_id": s["s1_scene_id"], "timestamp_ms": s["timestamp_ms"], "orbit_pass": s["orbit_pass"],
          "geometry": s["geometry"]} for s in scenes],
        crs="EPSG:4326",
    )

    grid = build_grid(bbox)
    # Celda x escena = una pasada de esfuerzo. Intersección por huella real de
    # la escena, no por bbox, para no sobrestimar la cobertura en los bordes.
    pases = gpd.sjoin(grid, scenes_gdf, how="inner", predicate="intersects")[
        ["cell_id", "s1_scene_id", "timestamp_ms", "orbit_pass"]
    ].drop_duplicates()

    exposure_panels[area_name] = pases
    out_path = f"esfuerzo_{area_name}.csv"
    pases.to_csv(out_path, index=False)
    print(f"  {len(scenes_gdf)} escenas -> {len(pases)} pares celda-escena. Guardado: {out_path}")

In [ ]:
from google.colab import files

print("Sube 'positivos_celda_escena.csv' (generado en el notebook de limpieza, sección 8.1) para completar el panel.")
uploaded = files.upload()
positives = pd.read_csv(list(uploaded.keys())[0])
positives["cell_id"] = positives["cell_id"].astype(str)

full_panels = []
for area_name, pases in exposure_panels.items():
    pos = positives.loc[positives["zona_estudio"] == area_name, ["cell_id", "s1_scene_id", "n_events"]]
    panel = pases.copy()
    panel["cell_id"] = panel["cell_id"].astype(str)
    panel = panel.merge(pos, on=["cell_id", "s1_scene_id"], how="left")
    panel["n_events"] = panel["n_events"].fillna(0).astype(int)
    panel["has_event"] = (panel["n_events"] > 0).astype(int)
    panel["zona_estudio"] = area_name
    full_panels.append(panel)

panel_completo = pd.concat(full_panels, ignore_index=True)
print(f"Panel celda-escena completo: {len(panel_completo)} filas, {panel_completo['has_event'].mean() * 100:.2f}% con detección")

# Agregado mensual por celda, listo como offset log(n_pases) en el modelo de la fase 6
panel_completo["fecha"] = pd.to_datetime(panel_completo["timestamp_ms"], unit="ms", utc=True)
panel_completo["mes"] = panel_completo["fecha"].dt.to_period("M").astype(str)

exposure_monthly = (
    panel_completo.groupby(["zona_estudio", "cell_id", "mes"])
    .agg(n_pases=("s1_scene_id", "nunique"), n_events=("n_events", "sum"))
    .reset_index()
)
exposure_monthly["tasa_deteccion"] = exposure_monthly["n_events"] / exposure_monthly["n_pases"]
exposure_monthly.to_csv("exposicion_mensual_celda.csv", index=False)

print(f"Guardado: exposicion_mensual_celda.csv ({len(exposure_monthly)} filas celda-mes)")
print("Entrada directa para sm.GLM(n_events, exog, family=Poisson(), offset=np.log(n_pases)) — fase 6 de la metodología.")

files.download("exposicion_mensual_celda.csv")

## 10. Viento: primera variable explicativa externa

El viento condiciona si una mancha es visible en SAR (mar en calma da falsos
positivos biogénicos; viento fuerte dispersa/borra la mancha), así que es la
variable externa con más prioridad antes de correr el modelo explicativo de la
fase 6.

**Nota sobre la fuente de datos — no es ERA5 "de verdad":** `ECMWF/ERA5/MONTHLY`
y `ECMWF/ERA5/DAILY` en Earth Engine dejaron de actualizarse en 2020 (verificado
directamente contra la API). `ECMWF/ERA5_LAND/MONTHLY_AGGR` sí llega a 2026,
pero está enmascarado sobre mar abierto — el Golfo Pérsico es casi todo mar, así
que devuelve prácticamente todo nulo (comprobado con un punto en medio del
Golfo). Para tener viento real sobre agua en todo 2023-2026 sin depender de una
cuenta en el Copernicus Climate Data Store (que necesitaría una API key
separada), se usa **NOAA/GFS0P25** (el modelo operativo GFS, tomando
`forecast_hours=0`, que equivale al análisis) — mismo tipo de variable
(viento a 10 m, m/s) aunque no sea reanálisis ERA5 estricto. Si más adelante
consigues acceso a CDS/ERA5, esta celda es fácil de sustituir sin tocar el
resto del pipeline (la salida tiene las mismas columnas: `cell_id`, `mes`,
`u10`, `v10`, `wind_speed`).

In [ ]:
from shapely.geometry import Point

GFS_SCALE = 27830  # resolución nativa aprox. de GFS0P25 (0.25°)


def monthly_wind_points(aoi_geom, start, end):
    """wind_speed = media de las magnitudes instantáneas, no la magnitud del
    vector medio: promediar u,v primero y sacar la magnitud después
    subestima la velocidad real cuando el viento cambia de dirección en el
    mes (comprobado: ~8% de subestimación en un mes de prueba)."""
    coll = (ee.ImageCollection("NOAA/GFS0P25")
            .filterDate(start, end)
            .filter(ee.Filter.eq("forecast_hours", 0))
            .select(["u_component_of_wind_10m_above_ground", "v_component_of_wind_10m_above_ground"],
                    ["u10", "v10"]))

    def add_speed(img):
        speed = img.expression("sqrt(u10*u10 + v10*v10)", {"u10": img.select("u10"), "v10": img.select("v10")})
        return img.addBands(speed.rename("wind_speed_inst"))

    img = coll.map(add_speed).select(["u10", "v10", "wind_speed_inst"]).mean().rename(["u10", "v10", "wind_speed"])
    return img.sample(region=aoi_geom, scale=GFS_SCALE, geometries=True).getInfo()


# Requiere el grid ya construido en esta sección (build_grid) y el mismo bbox
# de STUDY_AREAS. Si vienes de relanzar sección 9, `grid` ya existe; si no,
# se reconstruye aquí para la zona configurada.
viento_rows = []
for area_name, bbox in STUDY_AREAS.items():
    aoi = ee.Geometry.Rectangle(list(bbox))
    grid = build_grid(bbox)
    grid["centroid"] = grid.geometry.centroid
    grid_pts = gpd.GeoDataFrame(grid[["cell_id"]], geometry=grid["centroid"], crs=grid.crs)

    meses = pd.period_range("2023-01", pd.Timestamp.now(tz="UTC").to_period("M"), freq="M")
    for period in meses:
        start = period.start_time.strftime("%Y-%m-%d")
        end = (period.end_time + pd.Timedelta(days=1)).strftime("%Y-%m-%d")
        try:
            info = monthly_wind_points(aoi, start, end)
        except Exception as e:
            print(f"  [ERROR] {area_name} {period}: {e}")
            continue

        feats = info.get("features", [])
        if not feats:
            print(f"  {area_name} {period}: sin puntos de viento (omitido)")
            continue

        wind_pts = gpd.GeoDataFrame(
            [{"u10": f["properties"].get("u10"), "v10": f["properties"].get("v10"),
              "wind_speed": f["properties"].get("wind_speed"),
              "geometry": Point(f["geometry"]["coordinates"])} for f in feats if f.get("geometry")],
            crs="EPSG:4326",
        )
        joined = gpd.sjoin_nearest(grid_pts, wind_pts, how="left")[["cell_id", "u10", "v10", "wind_speed"]]
        joined["mes"] = str(period)
        joined["zona_estudio"] = area_name
        viento_rows.append(joined)

    print(f"{area_name}: viento calculado para {len(meses)} meses")

viento_celda_mes = pd.concat(viento_rows, ignore_index=True)
viento_celda_mes.to_csv("viento_celda_mes.csv", index=False)
print(f"Guardado: viento_celda_mes.csv ({len(viento_celda_mes)} filas)")

# Fusión con el panel de exposición ya generado en la sección 9
if "exposure_monthly" in globals():
    exposure_monthly["cell_id"] = exposure_monthly["cell_id"].astype(str)
    viento_celda_mes["cell_id"] = viento_celda_mes["cell_id"].astype(str)
    exposicion_con_viento = exposure_monthly.merge(
        viento_celda_mes, on=["zona_estudio", "cell_id", "mes"], how="left"
    )
    exposicion_con_viento.to_csv("exposicion_mensual_celda_con_viento.csv", index=False)
    print(f"Guardado: exposicion_mensual_celda_con_viento.csv ({len(exposicion_con_viento)} filas, "
          f"{exposicion_con_viento['wind_speed'].isna().mean() * 100:.2f}% sin viento asignado)")
else:
    print("[AVISO] 'exposure_monthly' no está en memoria (corre antes la sección 9). "
          "viento_celda_mes.csv se puede fusionar más tarde por ['zona_estudio','cell_id','mes'].")

## 11. Distancia a costa y a ruta marítima + GLM de la fase 6

Dos covariables espaciales estáticas por celda (no cambian mes a mes):

- **Distancia a costa**: desde el litoral de Natural Earth (10m coastline).
- **Distancia a la ruta/esquema de separación de tráfico (TSS) marítimo más cercano**: de OpenStreetMap vía Overpass. **No es densidad de tráfico AIS real** — se intentó primero el dataset "Global Shipping Traffic Density" del World Bank (su web estaba caída con error 5xx al construir este notebook, y el servicio ArcGIS que lo sirve solo expone teselas renderizadas, no valores de píxel) y Global Fishing Watch (requiere token de API). Si consigues acceso a cualquiera de los dos más adelante, sustituye esta celda sin tocar el resto — la salida tiene las mismas columnas (`cell_id`, `dist_costa_km`, `dist_ruta_maritima_km` → cambiarías la segunda por una densidad real).

Con esas dos variables más el viento (sección 10) y el esfuerzo (sección 9), se ajusta el **modelo explicativo con offset** que responde a las preguntas 3 (sesgo de detección) y 1 (estacionalidad) de la hoja de ruta: Poisson y binomial negativa sobre `n_events`, con `offset = log(n_pases)`.

In [ ]:
!pip -q install statsmodels

import requests as req
import numpy as np
import statsmodels.api as sm
from statsmodels.discrete.discrete_model import NegativeBinomial

UTM_CRS = "EPSG:32640"  # UTM 40N, razonable para todo el Golfo Pérsico

# --- Litoral (Natural Earth 10m coastline) ---
coast_resp = req.get(
    "https://raw.githubusercontent.com/nvkelso/natural-earth-vector/master/geojson/ne_10m_coastline.geojson",
    timeout=60,
)
with open("ne_10m_coastline.geojson", "wb") as f:
    f.write(coast_resp.content)


def fetch_shipping_lanes(bbox):
    minx, miny, maxx, maxy = bbox
    query = (
        f'[out:json][timeout:90];'
        f'(way["seamark:type"~"separation_lane|separation_zone|separation_boundary|fairway"]'
        f'({miny},{minx},{maxy},{maxx}););out geom;'
    )
    r = req.post("https://overpass-api.de/api/interpreter", data={"data": query},
                 timeout=120, headers={"User-Agent": "tfm-vertidos-research/1.0"})
    r.raise_for_status()
    data = r.json()
    from shapely.geometry import shape as shp
    lines = []
    for el in data.get("elements", []):
        coords = [(pt["lon"], pt["lat"]) for pt in el.get("geometry", [])]
        if len(coords) >= 2:
            lines.append(shp({"type": "LineString", "coordinates": coords}))
    return gpd.GeoDataFrame({"geometry": lines}, crs="EPSG:4326")


covariables_rows = []
for area_name, bbox in STUDY_AREAS.items():
    minx, miny, maxx, maxy = bbox
    grid = build_grid(bbox)
    grid["centroid"] = grid.geometry.centroid
    grid_pts = gpd.GeoDataFrame(grid[["cell_id"]], geometry=grid["centroid"], crs=grid.crs).to_crs(UTM_CRS)

    margin = 2.0
    coast = gpd.read_file("ne_10m_coastline.geojson",
                           bbox=(minx - margin, miny - margin, maxx + margin, maxy + margin))
    coast_union = coast.to_crs(UTM_CRS).union_all()
    grid_pts["dist_costa_km"] = grid_pts.geometry.distance(coast_union) / 1000.0

    lanes = fetch_shipping_lanes(bbox)
    if not lanes.empty:
        lanes_union = lanes.to_crs(UTM_CRS).union_all()
        grid_pts["dist_ruta_maritima_km"] = grid_pts.geometry.distance(lanes_union) / 1000.0
    else:
        grid_pts["dist_ruta_maritima_km"] = np.nan

    grid_pts["zona_estudio"] = area_name
    covariables_rows.append(grid_pts[["zona_estudio", "cell_id", "dist_costa_km", "dist_ruta_maritima_km"]])
    print(f"{area_name}: dist_costa media={grid_pts['dist_costa_km'].mean():.1f} km, "
          f"dist_ruta media={grid_pts['dist_ruta_maritima_km'].mean():.1f} km")

covariables_espaciales = pd.concat(covariables_rows, ignore_index=True)
covariables_espaciales.to_csv("covariables_espaciales_celda.csv", index=False)

# --- Panel final para el modelo: esfuerzo + viento + distancias ---
panel = exposicion_con_viento.copy()
panel["cell_id"] = panel["cell_id"].astype(str)
covariables_espaciales["cell_id"] = covariables_espaciales["cell_id"].astype(str)
panel_modelo = panel.merge(covariables_espaciales, on=["zona_estudio", "cell_id"], how="left")
panel_modelo.to_csv("panel_modelo_fase6.csv", index=False)
print(f"\npanel_modelo_fase6.csv: {len(panel_modelo)} filas")

# --- GLM con offset de esfuerzo ---
df = panel_modelo.copy()
df["anio"] = df["mes"].str.slice(0, 4).astype(int)
df["mes_num"] = df["mes"].str.slice(5, 7).astype(int)
df["mes_sin"] = np.sin(2 * np.pi * df["mes_num"] / 12)
df["mes_cos"] = np.cos(2 * np.pi * df["mes_num"] / 12)
df["anios_desde_inicio"] = df["anio"] - df["anio"].min() + (df["mes_num"] - 1) / 12.0

feature_cols = ["wind_speed", "dist_costa_km", "dist_ruta_maritima_km", "mes_sin", "mes_cos", "anios_desde_inicio"]
df = df[df["n_pases"] > 0].dropna(subset=feature_cols + ["n_events", "n_pases"]).copy()

X = df[feature_cols].copy()
for col in ["wind_speed", "dist_costa_km", "dist_ruta_maritima_km", "anios_desde_inicio"]:
    X[col] = (X[col] - X[col].mean()) / X[col].std()
X = sm.add_constant(X)
y = df["n_events"].values
offset = np.log(df["n_pases"].values)

poisson_res = sm.GLM(y, X, family=sm.families.Poisson(), offset=offset).fit()
print(poisson_res.summary())
dispersion = poisson_res.pearson_chi2 / poisson_res.df_resid
print(f"\nDispersión (Pearson chi2/df): {dispersion:.2f} (>1.5-2 = sobredispersión -> confiar más en la NB de abajo)")

nb_res = NegativeBinomial(y, X, exposure=df["n_pases"].values).fit(method="bfgs", maxiter=200, disp=False)
print(nb_res.summary())

rate_ratios = pd.DataFrame({"coef": nb_res.params, "rate_ratio": np.exp(nb_res.params), "p_value": nb_res.pvalues})
print("\nRate ratios (binomial negativa):")
print(rate_ratios.round(4))
rate_ratios.to_csv("glm_fase6_rate_ratios.csv")

## 12. Tráfico AIS real (Global Fishing Watch) y GLM final

Sustituye el proxy de la sección 11 (`dist_ruta_maritima_km`, distancia a rutas OSM) por presencia AIS real de Global Fishing Watch — mucho más informativo (en el modelo local resultó ser, junto con el viento, de las variables más fuertes).

**Necesitas tu propio token** de [globalfishingwatch.org/our-apis/tokens](https://globalfishingwatch.org/our-apis/tokens), guardado en los secretos de Colab como `GFW_API_TOKEN` (icono de la llave en la barra lateral izquierda → Add new secret). El token de la API es una credencial personal — no lo compartas ni lo pegues en el código.

**Aviso de la propia API:** el dataset de presencia AIS es de los más pesados; el token solo admite **un informe a la vez**. Si un mes falla, la celda reintenta con espera antes de seguir con el siguiente.

In [ ]:
import time
from google.colab import userdata

GFW_TOKEN = userdata.get("GFW_API_TOKEN")

!pip -q install gfwapiclient
import asyncio
import gfwapiclient as gfw_client


async def fetch_ais_month(client, geojson, start, end, max_retries=5, base_wait=20):
    for attempt in range(1, max_retries + 1):
        try:
            result = await client.fourwings.create_ais_presence_report(
                spatial_resolution="LOW", temporal_resolution="MONTHLY",
                start_date=start, end_date=end, geojson=geojson,
            )
            d = result.df()
            if d.empty:
                return pd.DataFrame(columns=["lat", "lon", "hours"])
            return d.groupby(["lat", "lon"], as_index=False)["hours"].sum()
        except Exception as e:
            wait = base_wait * attempt
            print(f"    intento {attempt}/{max_retries} falló ({type(e).__name__}); espero {wait}s...")
            await asyncio.sleep(wait)
    print(f"  [ERROR] {start}: agotados los reintentos")
    return None


async def build_ais_traffic():
    client = gfw_client.Client(access_token=GFW_TOKEN)
    ais_rows = []
    for area_name, bbox in STUDY_AREAS.items():
        minx, miny, maxx, maxy = bbox
        geojson = {"type": "Polygon", "coordinates": [[[minx, miny], [maxx, miny], [maxx, maxy], [minx, maxy], [minx, miny]]]}
        meses = pd.period_range("2023-01", pd.Timestamp.now(tz="UTC").to_period("M"), freq="M")
        for period in meses:
            start = period.start_time.strftime("%Y-%m-%d")
            end = (period.end_time + pd.Timedelta(days=1)).strftime("%Y-%m-%d")
            agg = await fetch_ais_month(client, geojson, start, end)
            if agg is None:
                continue
            agg["mes"] = str(period)
            agg["zona_estudio"] = area_name
            ais_rows.append(agg)
            print(f"  {area_name} {period}: {len(agg)} celdas GFW, {agg['hours'].sum():.0f} horas-buque")
            time.sleep(8)  # margen: el token solo admite 1 informe concurrente
    return pd.concat(ais_rows, ignore_index=True)


ais_raw = await build_ais_traffic()
ais_raw.to_csv("trafico_ais_gfw_celda01_mes.csv", index=False)

# Trasladar de la rejilla nativa de GFW (~0.1°) a nuestra rejilla fina por vecino más cercano
canon = ais_raw[["lat", "lon"]].drop_duplicates().reset_index(drop=True)
canon_pts = gpd.GeoDataFrame(canon, geometry=gpd.points_from_xy(canon["lon"], canon["lat"]), crs="EPSG:4326")

ais_traffic_rows = []
for area_name, bbox in STUDY_AREAS.items():
    grid = build_grid(bbox)
    grid["centroid"] = grid.geometry.centroid
    grid_pts = gpd.GeoDataFrame(grid[["cell_id"]], geometry=grid["centroid"], crs=grid.crs)
    nearest = gpd.sjoin_nearest(grid_pts, canon_pts, how="left")[["cell_id", "lat", "lon"]]
    fino = nearest.merge(ais_raw[ais_raw["zona_estudio"] == area_name], on=["lat", "lon"], how="left")
    fino["zona_estudio"] = area_name
    ais_traffic_rows.append(fino[["zona_estudio", "cell_id", "mes", "hours"]])

trafico_ais_celda_mes = pd.concat(ais_traffic_rows, ignore_index=True).rename(columns={"hours": "ais_hours"})
trafico_ais_celda_mes.to_csv("trafico_ais_celda_mes.csv", index=False)

# Panel final: sustituye dist_ruta_maritima_km por AIS real
panel_modelo["cell_id"] = panel_modelo["cell_id"].astype(str)
trafico_ais_celda_mes["cell_id"] = trafico_ais_celda_mes["cell_id"].astype(str)
panel_final = panel_modelo.merge(trafico_ais_celda_mes, on=["zona_estudio", "cell_id", "mes"], how="left")
panel_final["ais_hours"] = panel_final["ais_hours"].fillna(0.0)  # sin AIS detectado ese mes = 0 real, no vacío
panel_final.to_csv("panel_modelo_fase6_con_ais.csv", index=False)
print(f"\npanel_modelo_fase6_con_ais.csv: {len(panel_final)} filas, "
      f"{(panel_final['ais_hours'] == 0).mean() * 100:.1f}% celdas-mes sin tráfico detectado")

# --- GLM final: AIS real sustituye a la distancia a ruta OSM ---
df2 = panel_final.copy()
df2["anio"] = df2["mes"].str.slice(0, 4).astype(int)
df2["mes_num"] = df2["mes"].str.slice(5, 7).astype(int)
df2["mes_sin"] = np.sin(2 * np.pi * df2["mes_num"] / 12)
df2["mes_cos"] = np.cos(2 * np.pi * df2["mes_num"] / 12)
df2["anios_desde_2023"] = df2["anio"] - 2023 + (df2["mes_num"] - 1) / 12.0
df2["log_ais_hours"] = np.log1p(df2["ais_hours"])

feature_cols2 = ["wind_speed", "dist_costa_km", "log_ais_hours", "mes_sin", "mes_cos", "anios_desde_2023"]
df2 = df2[df2["n_pases"] > 0].dropna(subset=feature_cols2 + ["n_events", "n_pases"]).copy()

X2 = df2[feature_cols2].copy()
for col in ["wind_speed", "dist_costa_km", "log_ais_hours", "anios_desde_2023"]:
    X2[col] = (X2[col] - X2[col].mean()) / X2[col].std()
X2 = sm.add_constant(X2)
y2 = df2["n_events"].values
offset2 = np.log(df2["n_pases"].values)

nb_res2 = NegativeBinomial(y2, X2, exposure=df2["n_pases"].values).fit(method="bfgs", maxiter=200, disp=False)
print(nb_res2.summary())

rr2 = pd.DataFrame({"coef": nb_res2.params, "rate_ratio": np.exp(nb_res2.params), "p_value": nb_res2.pvalues})
print("\nRate ratios (binomial negativa, con AIS real):")
print(rr2.round(4))
rr2.to_csv("glm_fase6_con_ais_rate_ratios.csv")

## 13. Fase 7 — Modelo predictivo de probabilidad de detección

Responde a la pregunta 4 de la hoja de ruta. Es un problema de **evento raro**
(~5-8% de celdas-mes con detección), así que se evalúa con PR-AUC y Brier
score, no con accuracy. Validación **temporal**, no aleatoria: se ajusta con
2023 a mediados de 2025, se calibran las probabilidades con el resto de 2025,
y se valida con 2026 al completo — así el modelo nunca "ve" el futuro de la
misma celda en el mismo mes en que se evalúa.

**Aviso de calibración:** una primera versión con `class_weight="balanced"`
daba muy buena discriminación (ROC-AUC alto) pero un Brier score **peor** que
un modelo trivial que siempre predijera la tasa base — síntoma clásico de que
el reponderado de clases descalibra las probabilidades de un Random Forest.
Se corrigió quitando el reponderado y calibrando aparte con `CalibratedClassifierCV`
(isotónica) sobre un tramo de tiempo posterior al ajuste.

In [ ]:
!pip -q install -U scikit-learn  # FrozenEstimator requiere sklearn >= 1.6

from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.frozen import FrozenEstimator
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss, precision_recall_curve

dfp = panel_final.copy()
dfp["has_event"] = (dfp["n_events"] > 0).astype(int)
dfp["anio"] = dfp["mes"].str.slice(0, 4).astype(int)
dfp["mes_num"] = dfp["mes"].str.slice(5, 7).astype(int)
dfp["mes_sin"] = np.sin(2 * np.pi * dfp["mes_num"] / 12)
dfp["mes_cos"] = np.cos(2 * np.pi * dfp["mes_num"] / 12)
dfp["anios_desde_2023"] = dfp["anio"] - 2023 + (dfp["mes_num"] - 1) / 12.0
dfp["log_ais_hours"] = np.log1p(dfp["ais_hours"])

dfp = dfp.sort_values(["cell_id", "mes"]).reset_index(drop=True)
dfp["lag1_has_event"] = dfp.groupby("cell_id")["has_event"].shift(1).fillna(0)
dfp["lag1_n_events"] = dfp.groupby("cell_id")["n_events"].shift(1).fillna(0)

feature_cols = ["wind_speed", "dist_costa_km", "log_ais_hours", "n_pases",
                 "mes_sin", "mes_cos", "anios_desde_2023", "lag1_has_event", "lag1_n_events"]
dfp = dfp.dropna(subset=feature_cols + ["has_event"]).copy()

fit_ = dfp[dfp["mes"] < "2025-07"]
calib = dfp[(dfp["mes"] >= "2025-07") & (dfp["anio"] < 2026)]
test = dfp[dfp["anio"] >= 2026]
print(f"Ajuste: {len(fit_)} | Calibración: {len(calib)} | Test: {len(test)} "
      f"(tasas positivas: {fit_['has_event'].mean():.3f} / {calib['has_event'].mean():.3f} / {test['has_event'].mean():.3f})")

rf = RandomForestClassifier(n_estimators=400, max_depth=12, min_samples_leaf=20, n_jobs=-1, random_state=42)
rf.fit(fit_[feature_cols], fit_["has_event"])

rf_calibrado = CalibratedClassifierCV(FrozenEstimator(rf), method="isotonic")
rf_calibrado.fit(calib[feature_cols], calib["has_event"])

proba_test = rf_calibrado.predict_proba(test[feature_cols])[:, 1]
y_test = test["has_event"]

pr_auc = average_precision_score(y_test, proba_test)
roc_auc = roc_auc_score(y_test, proba_test)
brier = brier_score_loss(y_test, proba_test)
baseline_rate = y_test.mean()

print(f"\nPR-AUC:  {pr_auc:.4f}  (baseline aleatorio = {baseline_rate:.4f})")
print(f"ROC-AUC: {roc_auc:.4f}")
print(f"Brier:   {brier:.4f}  (baseline = {baseline_rate * (1 - baseline_rate):.4f})")

importancias = pd.DataFrame({"variable": feature_cols, "importancia": rf.feature_importances_}).sort_values("importancia", ascending=False)
print("\nImportancia de variables:")
print(importancias.to_string(index=False))

predicciones = test[["zona_estudio", "cell_id", "mes", "has_event"]].copy()
predicciones["proba_prevista"] = proba_test
predicciones.to_csv("fase7_predicciones_2026.csv", index=False)
importancias.to_csv("fase7_importancia_variables.csv", index=False)
print("\nGuardado: fase7_predicciones_2026.csv, fase7_importancia_variables.csv")
print("\nEste 'proba_prevista' por celda-mes es directamente el mapa de riesgo de la fase 8 (dashboard).")

---
### Notas

- El dataset subido cubre el **Golfo Pérsico** (bbox aprox. lon 48.16–56.98, lat 23.99–30.20). Si también tenés manchas del **Mar Rojo**, subí ese `geojson` por separado (o concatenalo con `slicks`) y volvé a correr desde la sección 3 — el código no asume una región fija.
- Sentinel-2 tiene datos globales consistentes desde 2015-2019 en adelante, y buena cobertura del Golfo Pérsico desde 2016. Landsat 8/9 sirve como respaldo cuando S2 está muy nublado o falta revisita cercana.
- Si `s2_image_id`/`landsat_image_id` salen `None` para muchas escenas, subí `MAX_CLOUD_S2`/`MAX_CLOUD_L8` o `DAYS_WINDOW`.
- Earth Engine tiene límites de requests; si el loop es muy lento, corré por lotes (`N_TEST`) o paralelizá con cuidado de no exceder cuota.
- **Sección 9 (catálogo de esfuerzo):** el `STUDY_AREAS`/`CELL_KM` de esa sección son una copia local — si cambiás el tamaño de celda o el bbox en el notebook de limpieza, actualizalos aquí también, o los `cell_id` dejan de ser comparables entre notebooks y el cruce con `positivos_celda_escena.csv` sale mal silenciosamente (no falla, simplemente no encuentra coincidencias).